# CNN Architecture Experiments on MNIST

The MNIST dataset contains 70,000 grayscale images of handwritten digits from 0 to 9, where each image is 28 by 28 pixels. It is a standard benchmark for image classification because the images are small, labeled, and easy to use for comparing model performance.

Convolutional Neural Networks (CNNs) usually outperform feedforward networks on image data because they preserve spatial structure, learn local patterns such as edges and curves, and reuse filters across the image. In contrast, a feedforward network flattens the image immediately, which removes useful spatial relationships between nearby pixels.

This notebook tests how CNN performance changes when the number of filters and kernel sizes are adjusted. It also compares the best CNN model against a simpler feedforward neural network to show why convolution is more effective for image classification.

In [ ]:
# Import the libraries required for modeling, analysis, and timing.
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for more reproducible results.
np.random.seed(42)
tf.random.set_seed(42)

# Load the MNIST dataset from Keras.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to the range [0, 1].
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape images to include a single channel dimension for CNN input.
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Print dataset shapes to confirm preprocessing.
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

## 3. Baseline CNN

The baseline CNN uses two convolutional layers to learn visual features from the digit images, and each convolution is followed by max pooling to reduce the feature map size. After feature extraction, the flattened output is passed through a dense hidden layer before the final softmax layer predicts one of the 10 digit classes.

In [ ]:
# Create a reusable function for building CNN models with different filters and kernels.
def build_cnn_model(filters=(32, 64), kernels=((3, 3), (3, 3)), input_shape=(28, 28, 1)):
    model = keras.Sequential([
        layers.Conv2D(filters[0], kernels[0], activation="relu", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(filters[1], kernels[1], activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    return model

# Create a reusable function for compiling and training models.
def compile_and_train_model(model, x_data, y_data, epochs=10, validation_split=0.2, batch_size=128):
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    start_time = time.time()
    history = model.fit(
        x_data,
        y_data,
        epochs=epochs,
        validation_split=validation_split,
        batch_size=batch_size,
        verbose=1
    )
    training_time = time.time() - start_time
    return history, training_time

# Create a helper function to evaluate models on the test set.
def evaluate_model(model, x_data, y_data):
    test_loss, test_accuracy = model.evaluate(x_data, y_data, verbose=0)
    return test_loss, test_accuracy

# Create a helper function to format architecture text consistently.
def describe_cnn_architecture(filters, kernels):
    return (
        f"Conv2D{filters[0]} {kernels[0]} -> MaxPool -> "
        f"Conv2D{filters[1]} {kernels[1]} -> MaxPool -> Dense128"
    )

# Train the baseline CNN architecture.
baseline_filters = (32, 64)
baseline_kernels = ((3, 3), (3, 3))
baseline_model = build_cnn_model(filters=baseline_filters, kernels=baseline_kernels)
baseline_history, baseline_training_time = compile_and_train_model(
    baseline_model,
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

# Record the final training and validation accuracy.
baseline_train_accuracy = baseline_history.history["accuracy"][-1]
baseline_val_accuracy = baseline_history.history["val_accuracy"][-1]
baseline_test_loss, baseline_test_accuracy = evaluate_model(baseline_model, x_test, y_test)

print(f"Baseline training accuracy: {baseline_train_accuracy:.4f}")
print(f"Baseline validation accuracy: {baseline_val_accuracy:.4f}")
print(f"Baseline test accuracy: {baseline_test_accuracy:.4f}")
print(f"Baseline training time: {baseline_training_time:.2f} seconds")

## 4. Filter Experiments

Increasing the number of filters generally allows the CNN to learn a richer set of visual patterns, which can improve validation accuracy because the model has more capacity to detect useful features. Smaller filter counts train faster, but they may miss subtle details that help separate similar handwritten digits.

At the same time, larger models can show diminishing returns because MNIST is a relatively simple dataset. Once the model already captures the important digit patterns, adding more filters may only produce small gains while increasing training time and the risk of mild overfitting.

In [ ]:
# Store experiment results for later comparison.
all_cnn_results = []
filter_validation_accuracies = {
    "(32, 64)": baseline_val_accuracy
}

# Save the baseline result as part of the CNN experiment pool.
all_cnn_results.append({
    "label": "Baseline CNN",
    "filters": baseline_filters,
    "kernels": baseline_kernels,
    "architecture": describe_cnn_architecture(baseline_filters, baseline_kernels),
    "val_accuracy": baseline_val_accuracy,
    "test_accuracy": baseline_test_accuracy,
    "training_time": baseline_training_time
})

# Train additional CNNs with different filter sizes.
filter_configs = [(16, 32), (64, 128)]

for filters in filter_configs:
    model = build_cnn_model(filters=filters, kernels=baseline_kernels)
    history, training_time = compile_and_train_model(
        model,
        x_train,
        y_train,
        epochs=10,
        validation_split=0.2
    )
    final_val_accuracy = history.history["val_accuracy"][-1]
    test_loss, test_accuracy = evaluate_model(model, x_test, y_test)

    filter_validation_accuracies[str(filters)] = final_val_accuracy
    all_cnn_results.append({
        "label": f"CNN Filters {filters}",
        "filters": filters,
        "kernels": baseline_kernels,
        "architecture": describe_cnn_architecture(filters, baseline_kernels),
        "val_accuracy": final_val_accuracy,
        "test_accuracy": test_accuracy,
        "training_time": training_time
    })

print("Validation accuracies by filter configuration:")
print(filter_validation_accuracies)

pd.DataFrame(all_cnn_results)[["label", "filters", "kernels", "val_accuracy", "training_time"]]

## 5. Kernel Size Experiments

Kernel size changes how much of the image the CNN sees at once when extracting features. Smaller kernels such as 3x3 usually work well because they capture local details efficiently, while larger kernels such as 5x5 can capture broader patterns but may use more parameters and smooth away some fine detail.

The best kernel choice is usually the one that balances detail extraction and generalization. On MNIST, 3x3 kernels often perform strongly because the digits are made of compact strokes, so fine-grained local patterns are especially useful.

In [ ]:
# Store validation accuracies for kernel experiments.
kernel_validation_accuracies = {
    "(3x3, 3x3)": baseline_val_accuracy
}

# Train CNNs with different kernel configurations while keeping filters fixed.
kernel_configs = [((5, 5), (5, 5)), ((5, 5), (3, 3))]

for kernels in kernel_configs:
    model = build_cnn_model(filters=baseline_filters, kernels=kernels)
    history, training_time = compile_and_train_model(
        model,
        x_train,
        y_train,
        epochs=10,
        validation_split=0.2
    )
    final_val_accuracy = history.history["val_accuracy"][-1]
    test_loss, test_accuracy = evaluate_model(model, x_test, y_test)

    kernel_key = f"({kernels[0][0]}x{kernels[0][1]}, {kernels[1][0]}x{kernels[1][1]})"
    kernel_validation_accuracies[kernel_key] = final_val_accuracy
    all_cnn_results.append({
        "label": f"CNN Kernels {kernel_key}",
        "filters": baseline_filters,
        "kernels": kernels,
        "architecture": describe_cnn_architecture(baseline_filters, kernels),
        "val_accuracy": final_val_accuracy,
        "test_accuracy": test_accuracy,
        "training_time": training_time
    })

print("Validation accuracies by kernel configuration:")
print(kernel_validation_accuracies)

pd.DataFrame(all_cnn_results)[["label", "filters", "kernels", "val_accuracy", "training_time"]]

## 6. Best Model Training and Plots

The best architecture is selected using the highest validation accuracy from the filter and kernel experiments. The selected model is then trained for 15 epochs so its learning behavior can be visualized with accuracy and loss curves.

If training accuracy continues to increase while validation accuracy levels off and validation loss starts rising, the model is beginning to overfit. If both training and validation accuracy improve together and the validation loss stays stable, the model is learning well and generalizing effectively.

In [ ]:
# Select the best CNN architecture based on validation accuracy.
best_cnn_config = max(all_cnn_results, key=lambda result: result["val_accuracy"])
best_filters = best_cnn_config["filters"]
best_kernels = best_cnn_config["kernels"]

print("Best architecture selected from experiments:")
print(best_cnn_config["architecture"])
print(f"Validation accuracy: {best_cnn_config['val_accuracy']:.4f}")

# Retrain the best architecture for more epochs to inspect learning curves.
best_model = build_cnn_model(filters=best_filters, kernels=best_kernels)
best_history, best_training_time = compile_and_train_model(
    best_model,
    x_train,
    y_train,
    epochs=15,
    validation_split=0.2
)
best_test_loss, best_test_accuracy = evaluate_model(best_model, x_test, y_test)

print(f"Best model test accuracy: {best_test_accuracy:.4f}")
print(f"Best model training time: {best_training_time:.2f} seconds")

# Print the model summary for the best architecture.
best_model.summary()

In [ ]:
# Plot training and validation accuracy across epochs.
plt.figure(figsize=(10, 4))
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Best CNN Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Plot training and validation loss across epochs.
plt.figure(figsize=(10, 4))
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.title("Best CNN Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Feedforward Network

A feedforward neural network is included as a comparison point because it uses the same data without convolution. This helps show how much performance is gained when the model can learn spatial image features directly.

In [ ]:
# Create a reusable function for the feedforward neural network.
def build_feedforward_model(input_shape=(28, 28, 1)):
    model = keras.Sequential([
        layers.Flatten(input_shape=input_shape),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    return model

# Train the feedforward model for comparison.
feedforward_model = build_feedforward_model()
feedforward_history, feedforward_training_time = compile_and_train_model(
    feedforward_model,
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)
feedforward_train_accuracy = feedforward_history.history["accuracy"][-1]
feedforward_val_accuracy = feedforward_history.history["val_accuracy"][-1]
feedforward_test_loss, feedforward_test_accuracy = evaluate_model(feedforward_model, x_test, y_test)

print(f"Feedforward training accuracy: {feedforward_train_accuracy:.4f}")
print(f"Feedforward validation accuracy: {feedforward_val_accuracy:.4f}")
print(f"Feedforward test accuracy: {feedforward_test_accuracy:.4f}")
print(f"Feedforward training time: {feedforward_training_time:.2f} seconds")

## 8. Performance Comparison Table

The CNN models are expected to outperform the feedforward network because convolution preserves local structure and learns features such as edges, curves, and strokes directly from the image. A feedforward network can still classify digits reasonably well, but it treats each pixel more independently after flattening, which makes feature learning less efficient.

CNNs work better for images because they use shared filters and local receptive fields, which reduces the number of parameters while improving feature extraction. This design makes CNNs more accurate and more suitable for visual pattern recognition tasks such as handwritten digit classification.

In [ ]:
# Create a comparison table for the main models in this assignment.
comparison_df = pd.DataFrame([
    {
        "Model": "Baseline CNN",
        "Architecture": describe_cnn_architecture(baseline_filters, baseline_kernels),
        "Test Accuracy": baseline_test_accuracy,
        "Training Time": baseline_training_time
    },
    {
        "Model": "Best CNN",
        "Architecture": describe_cnn_architecture(best_filters, best_kernels),
        "Test Accuracy": best_test_accuracy,
        "Training Time": best_training_time
    },
    {
        "Model": "Feedforward NN",
        "Architecture": "Flatten -> Dense128 -> Dense64 -> Dense10",
        "Test Accuracy": feedforward_test_accuracy,
        "Training Time": feedforward_training_time
    }
])

# Format the numeric output to make the table easy to read.
comparison_df["Test Accuracy"] = comparison_df["Test Accuracy"].round(4)
comparison_df["Training Time"] = comparison_df["Training Time"].round(2)

comparison_df

## 9. Final Reflection

If my final project involves image data, CNNs would likely improve performance because they are designed to capture spatial patterns efficiently. If the project involves sequential data such as time series or text, an RNN or another sequence-based model would be more appropriate because it can model dependencies across time or token order. If the final project is mostly tabular data, CNNs and RNNs may not add much value because traditional dense networks or tree-based models often handle tabular features more effectively. The best model choice depends on whether the data has spatial structure, sequential structure, or neither.

## 10. Clean Code Notes

This notebook uses reusable functions to build and train models, includes comments for each major step, follows consistent variable naming, and keeps each experiment in a clearly labeled section. The structure is designed to stay readable while still showing the full workflow from data loading to final comparison.

## 11. Bonus: Confusion Matrix and Test Set Evaluation

The confusion matrix shows which digits are classified correctly and where the best CNN still makes mistakes. This provides a more detailed view of performance than overall accuracy alone.

In [ ]:
# Generate predictions from the best CNN on the test set.
best_predictions = best_model.predict(x_test, verbose=0)
best_predicted_labels = np.argmax(best_predictions, axis=1)

# Create and visualize the confusion matrix.
confusion_matrix = tf.math.confusion_matrix(y_test, best_predicted_labels, num_classes=10).numpy()

plt.figure(figsize=(8, 6))
plt.imshow(confusion_matrix, cmap="Blues")
plt.title("Confusion Matrix for Best CNN")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(10))
plt.yticks(np.arange(10))
plt.colorbar()

for i in range(confusion_matrix.shape[0]):
    for j in range(confusion_matrix.shape[1]):
        plt.text(j, i, confusion_matrix[i, j], ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Best CNN final test accuracy: {best_test_accuracy:.4f}")
print(f"Best CNN final test loss: {best_test_loss:.4f}")